# Pancancer enrichment analysis step 2: Find enriched pathways
# Version 2: Using Reactome's Analysis Service API for enrichment analysis

## Setup

In [1]:
import cptac
import cptac.utils as ut
import pandas as pd
import numpy as np
import os
import datetime

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

STEP1_DIR = "step1_outputs"
STEP1_FILE = "tumor_change_20200529_195104_10000_perm.tsv"
STEP1_FILE_PATH = os.path.join(STEP1_DIR, STEP1_FILE)

STEP2_DIR = "step2_outputs_reactome"
STEP2_FILE = f"urls_{TIME_START}_from_{STEP1_FILE.rsplit('.', maxsplit=1)[0]}.tsv"
STEP2_FILE_PATH = os.path.join(STEP2_DIR, STEP2_FILE)

if not os.path.isdir(STEP2_DIR):
    os.mkdir(STEP2_DIR)

## Prepare data

In [3]:
# Read in the file from step 1
data = pd.read_csv(STEP1_FILE_PATH, sep='\t', index_col=0)

# The ranked enrichment analysis service wants ranking scores
# where "bigger is better", such as expression values or
# t scores. We are ranking by adjusted p-values, where smaller
# is better. So, we'll create a column of (1 - adj_p) to use
# for ranking.
data = data.assign(one_minus_adj_p=1 - data["adj_p"])

# Make a column where all increases are +1 and all decreases 
# are -1, because these are ratioed abundances, so we can't 
# compare magnitudes between genes--we can only compare whether 
# there was a change, and whether it was positive or negative
data = data.assign(simplified_change=data["change"])

data.loc[data["change"] > 0, "simplified_change"] = 1
data.loc[data["change"] < 0, "simplified_change"] = -1
data.loc[data["change"] == 0, "simplified_change"] = 0

In [4]:
data.head()

,cancer_type,protein,change,P_val,adj_p,reject_null,protein_str,one_minus_adj_p,simplified_change
0,ccrcc,"('A1BG', 'NP_570602.2')",0.282928,0.0000,0.00000,True,A1BG,1.00000,1.0
1,ccrcc,"('A1CF', 'NP_620310.1')",-0.551358,0.0000,0.00000,True,A1CF,1.00000,-1.0
2,ccrcc,"('A2M', 'NP_000005.2')",0.595512,0.0000,0.00000,True,A2M,1.00000,1.0
3,ccrcc,"('A4GALT', 'NP_001304967.1')",0.479410,0.1735,0.21431,False,A4GALT,0.78569,1.0
4,ccrcc,"('AAAS', 'NP_056480.1')",0.173579,0.0000,0.00000,True,AAAS,1.00000,1.0


## Run enrichment analysis

In [5]:
# For each cancer, find enriched pathways based on p value for differential expression   
all_enrichments = pd.DataFrame()

for cancer_type in data["cancer_type"].unique():
    
    ranked_data = data.loc[data["cancer_type"] == cancer_type, ["protein_str", "one_minus_adj_p"]].\
        set_index("protein_str").\
        rename(columns={"one_minus_adj_p": f"{cancer_type}_one_minus_adj_p"})

    unneeded_token, cancer_enriched = ut.reactome_enrichment_analysis(
        analysis_type="ranked", 
        data=ranked_data, 
        sort_by="ENTITIES_FDR", 
        ascending=True,
        include_high_level_diagrams=False, 
        include_interactors=False)
    
    # Record the cancer type and rename the pathway id column
    cancer_enriched = cancer_enriched.\
        assign(cancer_type=cancer_type).\
        rename(columns={"stId": "pathway_id"})
    
    # Append to the overall dataframe
    all_enrichments = all_enrichments.append(cancer_enriched)

In [6]:
all_enrichments.head()

,pathway_id,name,entities_ratio,entities_pValue,entities_fdr,entities_found,entities_total,cancer_type
0,R-HSA-3108214,SUMOylation of DNA damage response and repair ...,0.005581,0.045443,0.84264,81,81,ccrcc
1,R-HSA-4551638,SUMOylation of chromatin organization proteins,0.004272,0.071045,0.84264,62,62,ccrcc
2,R-HSA-4570464,SUMOylation of RNA binding proteins,0.003514,0.093002,0.84264,51,51,ccrcc
3,R-HSA-4615885,SUMOylation of DNA replication proteins,0.003445,0.095357,0.84264,50,50,ccrcc
4,R-HSA-159234,Transport of Mature mRNAs Derived from Intronl...,0.003238,0.102846,0.84264,47,47,ccrcc


In [7]:
all_enrichments.sort_values(by="entities_fdr")

,pathway_id,name,entities_ratio,entities_pValue,entities_fdr,entities_found,entities_total,cancer_type
0,R-HSA-6798695,Neutrophil degranulation,0.033072,6.216916e-12,1.500142e-08,434,480,colon
1,R-HSA-6791226,Major pathway of rRNA processing in the nucleo...,0.013022,2.825057e-07,1.376497e-04,181,189,colon
2,R-HSA-72203,Processing of Capped Intron-Containing Pre-mRNA,0.017638,2.855803e-07,1.376497e-04,233,256,colon
3,R-HSA-72163,mRNA Splicing - Major Pathway,0.012746,9.094579e-07,3.656021e-04,175,185,colon
4,R-HSA-72172,mRNA Splicing,0.013504,1.078818e-05,2.891233e-03,177,196,colon
9,R-HSA-927802,Nonsense-Mediated Decay (NMD),0.008543,1.856749e-04,2.580882e-02,114,124,colon
5,R-HSA-72689,Formation of a pool of free 40S subunits,0.007303,1.239514e-04,2.580882e-02,101,106,colon
6,R-HSA-72706,GTP hydrolysis and joining of the 60S ribosoma...,0.008268,1.212962e-04,2.580882e-02,112,120,colon
7,R-HSA-1799339,SRP-dependent cotranslational protein targetin...,0.008199,1.324252e-04,2.580882e-02,111,119,colon
8,R-HSA-156827,L13a-mediated translational silencing of Cerul...,0.008268,1.792866e-04,2.580882e-02,111,120,colon


## Summarize enrichment results from all cancers

In [8]:
# Make a table with a list of all pathways, and:
# - The number of cancer types for which that cancer type is enriched
# - The average of the adjusted p-values for that pathway
# - The average overlap proportion for that pathway
enrichment_summary = all_enrichments[["pathway_id", "name"]].drop_duplicates(keep="first")

times_enriched = enrichment_summary["pathway_id"].apply(
    lambda x: all_enrichments[all_enrichments["pathway_id"] == x].shape[0])

avg_fdr = enrichment_summary["pathway_id"].apply(
    lambda x: all_enrichments.loc[all_enrichments["pathway_id"] == x, "entities_fdr"].mean())

avg_overlap_props = enrichment_summary["pathway_id"].apply(
    lambda x: all_enrichments.loc[all_enrichments["pathway_id"] == x, "entities_ratio"].mean())

enrichment_summary = enrichment_summary.assign(
    times_enriched=times_enriched,
    pathway_avg_fdr=avg_fdr,
    pathway_avg_overlap=avg_overlap_props)

enrichment_summary = enrichment_summary.sort_values(
    by=["times_enriched", "pathway_avg_fdr"],
    ascending=[False, True])

enrichment_summary = enrichment_summary.reset_index(drop=True)

In [9]:
enrichment_summary.head()

,pathway_id,name,times_enriched,pathway_avg_fdr,pathway_avg_overlap
0,R-HSA-6798695,Neutrophil degranulation,8,0.587721,0.033072
1,R-HSA-6791226,Major pathway of rRNA processing in the nucleo...,8,0.725419,0.013022
2,R-HSA-72203,Processing of Capped Intron-Containing Pre-mRNA,8,0.725419,0.017638
3,R-HSA-72163,mRNA Splicing - Major Pathway,8,0.725447,0.012746
4,R-HSA-72172,mRNA Splicing,8,0.725763,0.013504


## Visualize pathways with expression change

In [10]:
# Submit the differential expression data for each cancer type to the analysis service, and get the tokens
# The data we'd submitted previously were the p values for the differential expression, but that's not what
# we want to visualize; we want to see whether the expression was up or down.
diff_expr_tokens = {}

for cancer_type in data["cancer_type"].unique():

    # Select data for that cancer type
    # Exclude samples where not reject_null
    omics = data.loc[(data["cancer_type"] == cancer_type) & data["reject_null"], 
                     ["protein_str", "simplified_change"]].\
        set_index("protein_str").\
        rename(columns={"simplified_change": f"{cancer_type}_simp_change"})

    analysis_token, unneeded_df = ut.reactome_enrichment_analysis(
        analysis_type="ranked", 
        data=omics, 
        sort_by="ENTITIES_FDR", 
        ascending=True,
        include_high_level_diagrams=True, 
        include_interactors=False)
    
    diff_expr_tokens[cancer_type] = analysis_token

In [11]:
# Select pathways to get URLs for
to_viz = enrichment_summary.iloc[0:10, :]

# Get URLs
id_list = []
cancer_type_list = []
expr_list = []
url_list = []

for idx in to_viz.index:
    pathway_id = to_viz.loc[idx, "pathway_id"]
    
    for cancer_type in data["cancer_type"].unique():
         
        # Get the URL
        expr_vals, url = ut.reactome_pathway_overlay(
            pathway=pathway_id,
            analysis_token=diff_expr_tokens[cancer_type],
            open_browser=False)
        
        id_list.append(pathway_id)
        cancer_type_list.append(cancer_type)
        expr_list.append(expr_vals[0])
        url_list.append(url)

urls_df = pd.DataFrame({
        "pathway_id": id_list,
        "cancer_type": cancer_type_list,
        "mean_expr": expr_list,
        "url": url_list})

In [12]:
# Join in the enrichment summary table
urls_df = urls_df.merge(
    enrichment_summary,
    how="left",
    left_on="pathway_id",
    right_on="pathway_id",
    validate="many_to_one")

urls_df

,pathway_id,cancer_type,mean_expr,url,name,times_enriched,pathway_avg_fdr,pathway_avg_overlap
0,R-HSA-6798695,ccrcc,0.233503,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.033072
1,R-HSA-6798695,colon,-0.134831,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.033072
2,R-HSA-6798695,endometrial,0.288515,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.033072
3,R-HSA-6798695,gbm,0.166227,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.033072
4,R-HSA-6798695,hnscc,0.133159,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.033072
5,R-HSA-6798695,lscc,-0.407595,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.033072
6,R-HSA-6798695,luad,-0.410000,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.033072
7,R-HSA-6798695,ovarian,-0.303754,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.033072
8,R-HSA-6791226,ccrcc,0.699422,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Major pathway of rRNA processing in the nucleo...,8,0.725419,0.013022
9,R-HSA-6791226,colon,0.771429,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Major pathway of rRNA processing in the nucleo...,8,0.725419,0.013022


In [13]:
# Join in the original enrichment data for each pathway and cancer type
urls_df = urls_df.merge(
    all_enrichments,
    how="left",
    left_on=["pathway_id", "name", "cancer_type"],
    right_on=["pathway_id", "name", "cancer_type"])

urls_df

,pathway_id,cancer_type,mean_expr,url,name,times_enriched,pathway_avg_fdr,pathway_avg_overlap,entities_ratio,entities_pValue,entities_fdr,entities_found,entities_total
0,R-HSA-6798695,ccrcc,0.233503,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.033072,0.033072,3.467008e-03,8.426404e-01,448,480
1,R-HSA-6798695,colon,-0.134831,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.033072,0.033072,6.216916e-12,1.500142e-08,434,480
2,R-HSA-6798695,endometrial,0.288515,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.033072,0.033072,1.403496e-03,8.219205e-01,450,480
3,R-HSA-6798695,gbm,0.166227,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.033072,0.033072,2.129677e-03,8.037977e-01,448,480
4,R-HSA-6798695,hnscc,0.133159,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.033072,0.033072,3.649260e-03,8.329209e-01,462,480
5,R-HSA-6798695,lscc,-0.407595,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.033072,0.033072,1.637558e-03,8.726502e-01,443,480
6,R-HSA-6798695,luad,-0.410000,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.033072,0.033072,1.800932e-04,4.412282e-01,441,480
7,R-HSA-6798695,ovarian,-0.303754,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.033072,0.033072,3.532076e-05,8.660650e-02,441,480
8,R-HSA-6791226,ccrcc,0.699422,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Major pathway of rRNA processing in the nucleo...,8,0.725419,0.013022,0.013022,3.132093e-02,8.426404e-01,179,189
9,R-HSA-6791226,colon,0.771429,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Major pathway of rRNA processing in the nucleo...,8,0.725419,0.013022,0.013022,2.825057e-07,1.376497e-04,181,189


In [14]:
# Print the dataframe in such a way that the links are clickable
urls_df.style.format('<a href="{0}">{0}</a>', subset="url")

,pathway_id,cancer_type,mean_expr,url,name,times_enriched,pathway_avg_fdr,pathway_avg_overlap,entities_ratio,entities_pValue,entities_fdr,entities_found,entities_total
0,R-HSA-6798695,ccrcc,0.233503,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDYxODI3MTZfMjEwODQ%3D,Neutrophil degranulation,8,0.587721,0.033072,0.033072,0.003467,0.842640,448,480
1,R-HSA-6798695,colon,-0.134831,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDYxODI3MjBfMjEwODU%3D,Neutrophil degranulation,8,0.587721,0.033072,0.033072,0.000000,0.000000,434,480
2,R-HSA-6798695,endometrial,0.288515,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDYxODI3MjRfMjEwODY%3D,Neutrophil degranulation,8,0.587721,0.033072,0.033072,0.001403,0.821921,450,480
3,R-HSA-6798695,gbm,0.166227,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDYxODI3MjhfMjEwODc%3D,Neutrophil degranulation,8,0.587721,0.033072,0.033072,0.002130,0.803798,448,480
4,R-HSA-6798695,hnscc,0.133159,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDYxODI3MzFfMjEwODg%3D,Neutrophil degranulation,8,0.587721,0.033072,0.033072,0.003649,0.832921,462,480
5,R-HSA-6798695,lscc,-0.407595,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDYxODI3MzVfMjEwODk%3D,Neutrophil degranulation,8,0.587721,0.033072,0.033072,0.001638,0.872650,443,480
6,R-HSA-6798695,luad,-0.410000,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDYxODI3MzlfMjEwOTA%3D,Neutrophil degranulation,8,0.587721,0.033072,0.033072,0.000180,0.441228,441,480
7,R-HSA-6798695,ovarian,-0.303754,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDYxODI3NDNfMjEwOTE%3D,Neutrophil degranulation,8,0.587721,0.033072,0.033072,0.000035,0.086606,441,480
8,R-HSA-6791226,ccrcc,0.699422,https://reactome.org/PathwayBrowser/#/R-HSA-6791226&DTAB=AN&ANALYSIS=MjAyMDA2MDYxODI3MTZfMjEwODQ%3D,Major pathway of rRNA processing in the nucleolus and cytosol,8,0.725419,0.013022,0.013022,0.031321,0.842640,179,189
9,R-HSA-6791226,colon,0.771429,https://reactome.org/PathwayBrowser/#/R-HSA-6791226&DTAB=AN&ANALYSIS=MjAyMDA2MDYxODI3MjBfMjEwODU%3D,Major pathway of rRNA processing in the nucleolus and cytosol,8,0.725419,0.013022,0.013022,0.000000,0.000138,181,189


In [15]:
# import webbrowser
# for url in urls_df["url"][0:8]:
#     webbrowser.open(url)

In [16]:
# Save the results
urls_df.to_csv(STEP2_FILE_PATH, sep='\t')